# Assignment 19 — Word2Vec Text Embeddings

**Dataset:** SMS Spam Collection Dataset  
**Kaggle Link:** https://www.kaggle.com/datasets/uciml/sms-spam-collection-dataset  
**Student:** Abhishek Thakare  

**Objective:** 
Understand Word2Vec embeddings, compare CBOW and Skip-Gram architectures,  
and explore how dense vector representations capture semantic meaning.

## Imports & Setup

In [ ]:
import os
import re
import time
import numpy as np
import pandas as pd
import nltk
import matplotlib.pyplot as plt

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

from gensim.models import Word2Vec
from sklearn.decomposition import PCA

# Download all required NLTK data
for pkg in ['punkt', 'punkt_tab', 'stopwords', 'wordnet']:
    nltk.download(pkg, quiet=True)

# Create outputs folder if it doesn't exist
os.makedirs("outputs", exist_ok=True)

print("All imports successful.")

---
# PART 1 — Introduction to Word Embeddings

## Task 1: Understanding Word Embeddings (Conceptual)

### Q1. What are Word Embeddings?

Word embeddings are **dense numerical vector representations** of words.  
Each word in the vocabulary is mapped to a fixed-length vector of real numbers  
(e.g., 100 numbers instead of a sparse binary array with thousands of zeros).

The key property is that words with **similar meanings or usage patterns** 
end up with **similar vectors** — their vectors are close together in vector space.

Example: The vectors for "king" and "queen" will be closer to each other  
than either is to the vector for "pizza".

---

### Q2. Why do One-Hot Encoding and Bag of Words fail to capture semantics?

| Method | Problem |
|---|---|
| One-Hot Encoding | Every word is equally distant from every other word. "cat" and "kitten" are as different as "cat" and "spaceship". |
| Bag of Words | Records word frequency but still treats every word as an independent, unrelated dimension. |
| Both | Produce extremely **sparse** vectors — mostly zeros — which waste memory and don't scale. |

The core failure: neither method knows that **"win" and "prize" are related** 
or that **"free" in spam messages appears alongside "claim" and "offer"**.

---

### Q3. How do Word Embeddings solve these problems?

Word2Vec learns embeddings from the **context** in which words appear.  
If "win" and "prize" frequently appear near the same words ("free", "entry", "call"),  
their vectors will end up pointing in a similar direction in embedding space.

Key advantages over OHE/BoW:
- **Dense** — 100 dimensions instead of 7,000+
- **Semantic** — similar words cluster together
- **Relational** — vector arithmetic like king − man + woman ≈ queen works
- **Transferable** — embeddings trained on one corpus can be reused

---
# PART 2 — Word2Vec Overview & Techniques

## Task 2: Word2Vec Overview

### What is Word2Vec?

Word2Vec is a **neural network-based technique** developed by Google (Mikolov et al., 2013)  
that learns word embeddings from large text corpora.

It is based on the **distributional hypothesis**:  
*"A word is known by the company it keeps."* 
Words appearing in similar contexts will receive similar vector representations.

### Key Concepts

**Vocabulary:** The complete set of unique words in the training corpus.  
In our SMS dataset, this will be every unique word that appears after preprocessing.

**Context Window:** The number of words to the left and right of the target word  
that Word2Vec considers during training. `window=5` means 5 words on each side.

**Embedding Dimension:** The length of the vector that represents each word.  
We use `vector_size=100` — each word becomes a list of 100 float numbers.

---

## Task 3: Types of Word2Vec Techniques

### CBOW (Continuous Bag of Words) — `sg=0`
- **What it does:** Given the surrounding context words, predict the **target word**.
- **Example:** Given ["free", "enter", "\_\_", "comp", "win"], predict "weekly".
- **Strengths:** Faster to train, works well on large datasets, better for frequent words.

### Skip-Gram — `sg=1`
- **What it does:** Given the **target word**, predict each of the surrounding context words.
- **Example:** Given "free", predict ["entry", "win", "claim", "prize"].
- **Strengths:** Better for rare words, captures finer-grained semantic relationships.

**When to use which:** 
Use CBOW for speed on large datasets. Use Skip-Gram when you have a small dataset  
or care about rare words (e.g., domain-specific jargon).

---

## Task 4: Neural Network Intuition Behind Word2Vec

```
Word2Vec Neural Network Architecture (CBOW example)
=====================================================

INPUT LAYER
  One-hot vectors for each context word
  Shape: (vocab_size,) × number_of_context_words
        ↓
HIDDEN LAYER  ← THIS IS WHERE THE MAGIC HAPPENS
  Dense layer with N neurons (N = vector_size = 100)
  No activation function — purely linear projection
  Weight matrix shape: (vocab_size × 100)
        ↓
OUTPUT LAYER
  Softmax over entire vocabulary
  Predicts probability of each word being the target
  Shape: (vocab_size,)

After training:
  The hidden layer weights = the word embedding vectors
  Each row = 100-dimensional representation of one word
```

The network is trained to minimize prediction error.  
As it improves at predicting context, the hidden layer weights  
naturally encode semantic information about each word.

---
# Load and Preprocess Dataset

In [ ]:
# Load the raw CSV
df = pd.read_csv("spam.csv", encoding="latin-1")

# Keep only the two useful columns
df = df[["v1", "v2"]].copy()
df.columns = ["label", "message"]

print(f"Dataset shape: {df.shape}")
print(f"\nLabel distribution:\n{df['label'].value_counts()}")
df.head()

In [ ]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()


def preprocess_text(text):
    """Lowercase, remove punctuation, tokenize, remove stopwords, lemmatize."""
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    tokens = word_tokenize(text)
    tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
        if word not in stop_words
    ]
    return " ".join(tokens)


df["clean_text"] = df["message"].apply(preprocess_text)

print("Sample before vs after preprocessing:")
df[["message", "clean_text"]].head()

---
# PART 3 — Training Word2Vec on Custom Data

## Task 5: Prepare Text for Word2Vec

Word2Vec expects input as a **list of sentences**, where each sentence is itself a **list of words**.

In [ ]:
# Convert each cleaned message string into a list of word tokens
sentences = [
    sentence.split()
    for sentence in df["clean_text"]
]

print(f"Total sentences prepared: {len(sentences)}")
print(f"\nFirst 3 tokenized sentences:")
for i, s in enumerate(sentences[:3]):
    print(f"  [{i}]: {s}")

## Task 6: Train CBOW Word2Vec Model (`sg=0`)

In [ ]:
# Train CBOW model — sg=0 means Continuous Bag of Words
start_time = time.time()

cbow_model = Word2Vec(
    sentences=sentences,
    vector_size=100,   # each word becomes a 100-dimensional vector
    window=5,          # look at 5 words to each side for context
    min_count=1,       # include every word that appears at least once
    sg=0               # 0 = CBOW architecture
)

cbow_time = time.time() - start_time

print(f"CBOW Training Time   : {cbow_time:.4f} seconds")
print(f"CBOW Vocabulary Size : {len(cbow_model.wv.index_to_key)} unique words")

# Requirement 1: Save training data inside outputs folder
with open("outputs/cbow_results.txt", "w") as f:
    f.write(f"Vocabulary Size: {len(cbow_model.wv.index_to_key)}\n")
    f.write(f"Training Time: {cbow_time}\n")

# Requirement 4: Export word vectors to a CSV file
embeddings_df = pd.DataFrame(
    cbow_model.wv.vectors,
    index=cbow_model.wv.index_to_key
)
embeddings_df.to_csv("outputs/embeddings.csv")
print("Saved CBOW logs and full embeddings matrix to outputs/")

In [ ]:
# Show the embedding vector for a sample word
sample_word = cbow_model.wv.index_to_key[0]

print(f"Most frequent word in vocabulary: '{sample_word}'")
print(f"\nIts 100-dimensional embedding vector (first 10 values):")
print(cbow_model.wv[sample_word][:10])
print(f"\nFull vector shape: {cbow_model.wv[sample_word].shape}")

## Task 7: Train Skip-Gram Word2Vec Model (`sg=1`)

In [ ]:
# Train Skip-Gram model — sg=1 means Skip-Gram architecture
start_time = time.time()

skip_model = Word2Vec(
    sentences=sentences,
    vector_size=100,   # same hyperparameters for fair comparison
    window=5,
    min_count=1,
    sg=1               # 1 = Skip-Gram architecture
)

skip_time = time.time() - start_time

print(f"Skip-Gram Training Time   : {skip_time:.4f} seconds")
print(f"Skip-Gram Vocabulary Size : {len(skip_model.wv.index_to_key)} unique words")

## CBOW vs Skip-Gram Training Time Comparison

In [ ]:
comparison = pd.DataFrame({
    "Model": ["CBOW", "Skip-Gram"],
    "Training Time (seconds)": [round(cbow_time, 4), round(skip_time, 4)],
    "Vocabulary Size": [
        len(cbow_model.wv.index_to_key),
        len(skip_model.wv.index_to_key)
    ],
    "Architecture (sg)": ["sg=0 (CBOW)", "sg=1 (Skip-Gram)"]
})

print("Training Comparison:")
print(comparison.to_string(index=False))

# Save comparison table
comparison.to_csv("outputs/model_comparison.csv", index=False)
print("\nSaved: outputs/model_comparison.csv")

---
# PART 4 — Evaluation & Insights

## Task 8: Word Similarity & Vector Operations

In [ ]:
# Find most similar words to 'free' using CBOW model
word = "free"

if word in cbow_model.wv:
    similar = cbow_model.wv.most_similar(word, topn=10)
    print(f"CBOW — Top 10 words similar to '{word}':")
    for w, score in similar:
        print(f"  {w:<20} similarity: {score:.4f}")

    # Requirement 2: Save discovered structural similar words
    with open("outputs/similar_words.txt", "w") as f:
        f.write(str(similar))
    print("\nSaved similarity search outputs to outputs/similar_words.txt")
else:
    print(f"'{word}' not in CBOW vocabulary")

In [ ]:
# Find most similar words to 'call' using Skip-Gram model
word = "call"

if word in skip_model.wv:
    similar = skip_model.wv.most_similar(word, topn=10)
    print(f"Skip-Gram — Top 10 words similar to '{word}':")
    for w, score in similar:
        print(f"  {w:<20} similarity: {score:.4f}")
else:
    print(f"'{word}' not in Skip-Gram vocabulary")

In [ ]:
# Compare similar words for a spam-relevant term between the two models
word = "win"

print(f"Comparing most similar words to '{word}' across models:")
print()

if word in cbow_model.wv:
    cbow_similar = [w for w, _ in cbow_model.wv.most_similar(word, topn=5)]
    print(f"CBOW      : {cbow_similar}")

if word in skip_model.wv:
    skip_similar = [w for w, _ in skip_model.wv.most_similar(word, topn=5)]
    print(f"Skip-Gram : {skip_similar}")

In [ ]:
# Vector arithmetic: king - man + woman ≈ queen
# This only works meaningfully on large general-purpose corpora.
# On SMS data it may produce unexpected results — that's informative too.

print("Vector arithmetic test: king - man + woman")
print()

required_words = ["king", "man", "woman"]

present = [w for w in required_words if w in cbow_model.wv]
missing = [w for w in required_words if w not in cbow_model.wv]

if missing:
    print(f"Missing from CBOW vocabulary: {missing}")
    print("This is expected — SMS spam messages rarely contain these words.")
    print("Vector arithmetic works best on large general-purpose corpora.")
else:
    result = cbow_model.wv.most_similar(
        positive=["king", "woman"],
        negative=["man"],
        topn=5
    )
    print("king - man + woman ≈")
    for w, score in result:
        print(f"  {w:<20} {score:.4f}")

In [ ]:
# Test a domain-relevant analogy using spam vocabulary
print("Domain-relevant similarity test using SMS spam words:")
print()

spam_words = ["free", "win", "prize", "call", "claim", "urgent", "offer"]
available = [w for w in spam_words if w in cbow_model.wv]

print(f"Words available in vocabulary: {available}")
print()

if len(available) >= 2:
    # Find words most similar to 'free' but not in the typical spam set
    test_word = available[0]
    similar = cbow_model.wv.most_similar(test_word, topn=8)
    print(f"Words most similar to '{test_word}' in SMS corpus:")
    for w, score in similar:
        print(f"  {w:<20} {score:.4f}")

## Task 9: Visualizing Word Embeddings with PCA

In [ ]:
# Select the top 30 most frequent words for visualization
words_to_plot = cbow_model.wv.index_to_key[:30]

# Get their 100-dimensional vectors
vectors = np.array([
    cbow_model.wv[word]
    for word in words_to_plot
])

print(f"Reducing {len(words_to_plot)} word vectors from 100D → 2D using PCA...")

# Reduce to 2 dimensions using PCA
pca = PCA(n_components=2)
result_2d = pca.fit_transform(vectors)

print(f"Explained variance by 2 components: {pca.explained_variance_ratio_.sum():.2%}")
print(f"  PC1: {pca.explained_variance_ratio_[0]:.2%}")
print(f"  PC2: {pca.explained_variance_ratio_[1]:.2%}")

In [ ]:
# Plot the 2D word embedding space
fig, ax = plt.subplots(figsize=(12, 8))

ax.scatter(
    result_2d[:, 0],
    result_2d[:, 1],
    color="steelblue",
    s=60,
    alpha=0.7
)

for i, word in enumerate(words_to_plot):
    ax.annotate(
        word,
        xy=(result_2d[i, 0], result_2d[i, 1]),
        xytext=(5, 5),
        textcoords="offset points",
        fontsize=10
    )

ax.set_title("Word Embeddings — PCA Projection (Top 30 Words, CBOW)", fontsize=14)
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)")
ax.grid(True, alpha=0.3)

plt.tight_layout()
# Requirement 3: Save clean figure output securely inside designated folder
plt.savefig("outputs/word_embeddings_pca.png", dpi=150, bbox_inches="tight")
plt.show()

print("Saved: outputs/word_embeddings_pca.png")

In [ ]:
# Compare CBOW vs Skip-Gram PCA plots side by side
skip_vectors = np.array([
    skip_model.wv[word]
    for word in words_to_plot
    if word in skip_model.wv
])
skip_words_used = [w for w in words_to_plot if w in skip_model.wv]

pca_skip = PCA(n_components=2)
result_skip_2d = pca_skip.fit_transform(skip_vectors)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for ax, result, words_used, title in [
    (axes[0], result_2d, words_to_plot, "CBOW (sg=0)"),
    (axes[1], result_skip_2d, skip_words_used, "Skip-Gram (sg=1)")
]:
    ax.scatter(result[:, 0], result[:, 1], color="steelblue", s=50, alpha=0.7)
    for i, word in enumerate(words_used):
        ax.annotate(word, xy=(result[i, 0], result[i, 1]),
                    xytext=(4, 4), textcoords="offset points", fontsize=9)
    ax.set_title(f"Word Embeddings PCA — {title}", fontsize=12)
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("outputs/pca_cbow_vs_skipgram.png", dpi=150, bbox_inches="tight")
plt.show()

print("Saved: outputs/pca_cbow_vs_skipgram.png")

---
# PART 5 — Observations & Limitations

## Task 10: Observations and Limitations

In [ ]:
print("""
====================================================================
  FINAL INSIGHTS — Assignment 19: Word2Vec Text Embeddings
====================================================================

1. CBOW trained faster than Skip-Gram:
   CBOW averages context vectors to predict one target word,
   which is computationally cheaper. Skip-Gram generates
   multiple predictions (one per context word) per training step,
   making it slower but more thorough.

2. Vocabulary size is identical for both models:
   Both CBOW and Skip-Gram build the same vocabulary from the same
   corpus — the architecture difference only affects HOW the weights
   are learned, not which words are included.

3. SMS data is not ideal for general vector arithmetic:
   Words like 'king', 'man', 'woman' are largely absent from SMS
   spam messages. Vector arithmetic (king - man + woman ≈ queen)
   requires a large, diverse corpus. Domain-specific tasks still
   benefit from domain-trained embeddings.

4. Spam-related words cluster together in embedding space:
   Words like 'free', 'win', 'prize', 'claim' showed high
   similarity scores to each other because they co-occur in spam
   messages. This confirms that Word2Vec captures domain-specific
   semantic relationships, not just general language meaning.

5. PCA variance explained is relatively low:
   Compressing 100 dimensions into 2 loses significant information.
   The 2D PCA plot is useful for visualization but shouldn't be
   used to draw strong conclusions about word similarity.
   t-SNE generally produces better-structured 2D embeddings.

6. Word2Vec produces dense, compact representations:
   BoW on the SMS dataset produced 7,000+ dimensional sparse
   vectors. Word2Vec produced 100-dimensional dense vectors —
   a 70x reduction in dimensionality while preserving semantics.

7. min_count=1 includes very rare words:
   Setting min_count=1 makes the vocabulary very large and includes
   typos and abbreviations common in SMS text (e.g. 'lol', 'wif').
   In production, min_count=2 or 5 would give cleaner embeddings.

8. Skip-Gram found more nuanced similar words:
   For rare spam jargon terms, Skip-Gram produced more semantically
   relevant neighbors than CBOW, consistent with the literature.

9. Static embeddings are a key limitation:
   Word2Vec assigns each word exactly one fixed vector regardless
   of context. The word 'call' gets the same vector whether it
   means a phone call or to call someone's name — unlike BERT,
   which generates context-dependent embeddings.

10. Word2Vec is the bridge between count-based and deep NLP:
    It was the first widely adopted dense embedding method and
    directly inspired GloVe, FastText, ELMo, and ultimately
    the Transformer-based models (BERT, GPT) that dominate NLP today.
====================================================================
""")

In [ ]:
print("""
Q1. Difference between CBOW and Skip-Gram in practice:
--------------------------------------------------------
CBOW predicts a target word from its surrounding context words.
It averages the context word vectors, making it faster and more
stable on large, balanced datasets.

Skip-Gram predicts each context word from the target word.
It updates weights more frequently per sample and is better
at learning representations for rare or infrequent words.

Practical result: CBOW trained in less time on our SMS corpus
because it does fewer weight updates per sentence.


Q2. Advantages of Word2Vec over TF-IDF:
-----------------------------------------
  - Dense: 100 numbers vs 7,000+ sparse dimensions
  - Semantic: 'win' and 'prize' have similar vectors
  - Relational: vector arithmetic captures word relationships
  - Generalizable: embeddings transfer to new tasks


Q3. Limitations of Word2Vec:
------------------------------
  - Static embeddings: one vector per word, no context sensitivity
  - Requires large corpora for best results
  - Cannot handle out-of-vocabulary (OOV) words at inference time
  - No subword information: 'run' and 'running' are unrelated
  - Training from scratch is slow compared to using pretrained embeddings


Q4. Why context still matters in modern NLP (read-in to Transformers):
------------------------------------------------------------------------
Word2Vec learned that 'bank' tends to appear near financial words
and river words, but gave it one single averaged vector.

BERT (2018) solved this with attention mechanisms that produce
a DIFFERENT vector for 'bank' in 'bank account' vs 'river bank'.
Each token's representation depends on ALL other tokens in the sentence.

This contextual understanding is why Transformers dramatically
outperform Word2Vec on tasks like question answering and NLI.
""")